# note

I FORGOT TO DO TRAIN TEST EVAL BRUH see v2 ig

# step 1: setup and imports

In [10]:
import pandas as pd
import numpy as np
import torch
from sklearn.model_selection import train_test_split
from transformers import BertTokenizer, BertForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import f1_score, roc_auc_score
from datasets import Dataset

# Emotion label columns — all 28 of them
EMOTION_COLS = [
    'admiration', 'amusement', 'anger', 'annoyance', 'approval', 'caring',
    'confusion', 'curiosity', 'desire', 'disappointment', 'disapproval',
    'disgust', 'embarrassment', 'excitement', 'fear', 'gratitude', 'grief',
    'joy', 'love', 'nervousness', 'optimism', 'pride', 'realization',
    'relief', 'remorse', 'sadness', 'surprise', 'neutral'
]

# Step 2 — Load & clean your data

In [ ]:
df = pd.read_csv("goemotions_cleaned.csv")

# Keep only what we need
df = df[['text'] + EMOTION_COLS]

# Drop rows with no emotion label at all
df = df[df[EMOTION_COLS].sum(axis=1) > 0].reset_index(drop=True)

print(f"Dataset size after cleaning: {len(df)} rows")  # ~57,157

# Your labels are already 0s and 1s — no encoding needed!

Dataset size after cleaning: 57157 rows


# Step 3 — Tokenize


In [12]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128  # Reddit comments are usually short
    )

# Step 4 — Build the dataset


In [13]:
# Split into train and test
train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)

train_dataset = Dataset.from_pandas(train_df.reset_index(drop=True))
test_dataset = Dataset.from_pandas(test_df.reset_index(drop=True))

# Tokenize both splits
train_dataset = train_dataset.map(tokenize, batched=True)
test_dataset = test_dataset.map(tokenize, batched=True)

# Tell the dataset which columns to use during training
# ⚠️ For multi-label, labels must be a float tensor, not int
def format_labels(example):
    example["labels"] = [float(example[col]) for col in EMOTION_COLS]
    return example

train_dataset = train_dataset.map(format_labels)
test_dataset = test_dataset.map(format_labels)

train_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
test_dataset.set_format("torch", columns=["input_ids", "attention_mask", "labels"])

Map: 100%|██████████| 11432/11432 [00:01<00:00, 10849.60 examples/s]


# Step 5 — Load the model (multi-label mode)


In [14]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=28,
    problem_type="multi_label_classification"  # ← this is the key difference!
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2577.21it/s, Materializing param=bert.pooler.dense.weight]                               
BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those p

# Step 6 — Evaluation metrics


In [15]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    # Convert logits to predictions using 0.5 threshold
    predictions = (torch.sigmoid(torch.tensor(logits)) > 0.5).int().numpy()
    labels = labels.astype(int)
    return {
        "f1_micro": f1_score(labels, predictions, average="micro", zero_division=0),
        "f1_macro": f1_score(labels, predictions, average="macro", zero_division=0),
    }

# Step 7 — Train


In [16]:
%pip install "accelerate>=1.1.0" -q


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [19]:
import torch
print(torch.cuda.is_available())

False


In [ ]:
training_args = TrainingArguments(
    output_dir="./goemotions_bert",
    num_train_epochs=3,
    per_device_train_batch_size=8,   # lower to 8 if you get out-of-memory errors
    per_device_eval_batch_size=8,
    learning_rate=2e-5,
    eval_strategy="epoch"
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_micro",
    logging_steps=100,
    warmup_steps=200,                 # helps BERT warm up gradually at the start
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    compute_metrics=compute_metrics,
)

trainer.train()

/Users/zacchaeus_tan/Documents/smu/dap/personal-learning-transformers/.venv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

# Step 8 — Make predictions on new text


- I was running code on collab notebook and using the GPU provided for training, however, here on VSC, I am using my CPU to train the model. 

- hence the added cell below is added when i got an error saying that the input tensors in the predict_emotion functions are on CPU, but they should be on the same device

- The key addition is inputs = {k: v.to(device) for k, v in inputs.items()} — this moves all the input tensors to the GPU to match where your model lives.

- since running on GPU, add fp16=True to your TrainingArguments — it'll make your training significantly faster!

In [ ]:
def predict_emotions(text, threshold=0.5):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.sigmoid(outputs.logits[0])
    results = {EMOTION_COLS[i]: round(probs[i].item(), 3) for i in range(28)}
    # Only show emotions above the threshold
    active = {k: v for k, v in results.items() if v >= threshold}
    return dict(sorted(active.items(), key=lambda x: -x[1]))

print(predict_emotions("I can't believe how amazing that was!"))
# → {'admiration': 0.82, 'excitement': 0.71, 'joy': 0.63}

print(predict_emotions("This makes me so angry and disappointed."))
# → {'anger': 0.79, 'disappointment': 0.68, 'annoyance': 0.54}

- the cell below is used when the above returns empty dictionaries

- this mean the model's confidence scores are all below the 0.5 threshold as the model has just started training

In [ ]:
# See what the raw probabilities actually look like
import torch

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def predict_emotions_debug(text, threshold = 0.2):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, max_length=128)
    inputs = {k: v.to(device) for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.sigmoid(outputs.logits[0])
    results = {EMOTION_COLS[i]: round(probs[i].item(), 3) for i in range(28)}
    active = {k: v for k, v in results.items() if v >= threshold}
    return dict(sorted(results.items(), key=lambda x: -x[1]))  # show ALL, no threshold

print(predict_emotions_debug("I can't believe how amazing that was!",threshold = 0.1))

- after running the above code, these were my top few emotions:
{'admiration': 0.355, 'surprise': 0.25, 'excitement': 0.134, 'joy': 0.075, }

- yay! the model is working as the emotions detected are correct however scores are just a low overall

- this is normal as i've only trained using 3 epochs

- can try to make the model more confident by increasing the number of epochs

# Step 9 — Save


In [ ]:
model.save_pretrained("./my_goemotions_model")
tokenizer.save_pretrained("./my_goemotions_model")